YOLO格式转换成COCO并验证

In [2]:
from pathlib import Path
from collections import OrderedDict
import json

# ============ 基础配置 ============
IMG_W, IMG_H = 640, 512
NUM_CLASSES = 4
CLASS_NAMES = ["Person", "Car", "Bicycle", "OtherVehicle"]

# ============ 路径配置 ============
ROOT = Path("/root/retest")
IMAGE_DIR = ROOT / "images"
LABEL_DIR = ROOT / "labels"

# COCO 输出目录
OUT_DIR = Path("/root/retest/baseline_results/fast_cnn_results/hit-uav_coco")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("YOLO 图片根目录:", IMAGE_DIR)
print("YOLO 标签根目录:", LABEL_DIR)
print("COCO 输出目录:", OUT_DIR)

YOLO 图片根目录: /root/retest/images
YOLO 标签根目录: /root/retest/labels
COCO 输出目录: /root/retest/baseline_results/fast_cnn_results/hit-uav_coco


In [3]:
def yolo_to_coco(root, split, img_w=640, img_h=512, num_classes=4):
    root = Path(root)
    image_dir = root / "images" / split
    label_dir = root / "labels" / split

    if not image_dir.exists():
        raise FileNotFoundError(f"图片目录不存在：{image_dir}")
    if not label_dir.exists():
        raise FileNotFoundError(f"标签目录不存在：{label_dir}")

    images = []
    annotations = []
    ann_id = 1
    n_oob = 0

    for img_id, img_path in enumerate(sorted(image_dir.glob("*.jpg")), start=1):
        stem = img_path.stem
        txt_path = label_dir / f"{stem}.txt"

        if not txt_path.exists():
            raise FileNotFoundError(f"缺少标签文件：{txt_path}")

        images.append({
            "id": img_id,
            "file_name": img_path.name,
            "width": img_w,
            "height": img_h,
        })

        if txt_path.stat().st_size == 0:
            continue

        for line in txt_path.read_text().splitlines():
            line = line.strip()
            if not line:
                continue

            parts = line.split()
            c = int(parts[0])
            if c < 0 or c >= num_classes:
                n_oob += 1
                continue

            cx, cy, w, h = map(float, parts[1:5])
            x = (cx - w / 2) * img_w
            y = (cy - h / 2) * img_h
            w *= img_w
            h *= img_h

            annotations.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": c + 1,
                "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                "area": round(w * h, 2),
                "iscrowd": 0,
            })
            ann_id += 1

    categories = [
        {"id": i + 1, "name": name, "supercategory": name}
        for i, name in enumerate(CLASS_NAMES)
    ]

    coco = {
        "images": images,
        "annotations": annotations,
        "categories": categories,
    }

    return coco, n_oob

In [4]:
for split in ["train", "val"]:
    coco, n_oob = yolo_to_coco(
        root=ROOT,
        split=split,
        img_w=IMG_W,
        img_h=IMG_H,
        num_classes=NUM_CLASSES,
    )

    out_path = OUT_DIR / f"{split}.json"
    out_path.write_text(json.dumps(coco, indent=2))

    print("=" * 50)
    print(f"split       : {split}")
    print(f"输出文件     : {out_path}")
    print(f"图片数       : {len(coco['images'])}")
    print(f"标注框数     : {len(coco['annotations'])}")
    print(f"过滤越界框   : {n_oob}")
    print(f"类别定义     : {[(c['id'], c['name']) for c in coco['categories']]}")

split       : train
输出文件     : /root/retest/baseline_results/fast_cnn_results/hit-uav_coco/train.json
图片数       : 2008
标注框数     : 17518
过滤越界框   : 110
类别定义     : [(1, 'Person'), (2, 'Car'), (3, 'Bicycle'), (4, 'OtherVehicle')]
split       : val
输出文件     : /root/retest/baseline_results/fast_cnn_results/hit-uav_coco/val.json
图片数       : 287
标注框数     : 2453
过滤越界框   : 7
类别定义     : [(1, 'Person'), (2, 'Car'), (3, 'Bicycle'), (4, 'OtherVehicle')]


In [19]:
from pathlib import Path
import json
import numpy as np

IMG_W, IMG_H = 640, 512
NUM_CLASSES = 4

ROOT = Path("/root/retest")
COCO_DIR = Path("/root/retest/baseline_results/fast_cnn_results/hit-uav_coco")

print("ROOT:", ROOT.exists())
print("COCO_DIR:", COCO_DIR.exists())
print("val.json:", (COCO_DIR / "val.json").exists())
print("train.json:", (COCO_DIR / "train.json").exists())

ROOT: True
COCO_DIR: True
val.json: True
train.json: True


In [20]:
def load_yolo_normalized(root, split):
    label_dir = Path(root) / "labels" / split
    data = {}

    for txt in sorted(label_dir.glob("*.txt")):
        items = []
        if txt.stat().st_size > 0:
            for line in txt.read_text().splitlines():
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                c = int(parts[0])
                if c < 0 or c >= NUM_CLASSES:
                    continue
                cx, cy, w, h = map(float, parts[1:5])
                cx = round(cx, 6)
                cy = round(cy, 6)
                w = round(w, 6)
                h = round(h, 6)
                if w <= 0 or h <= 0:
                    raise ValueError(f"非法 YOLO 框：{txt.name}")
                items.append((c, cx, cy, w, h))
        data[txt.stem] = items

    return data

In [21]:
def load_coco_and_roundtrip(coco_path):
    coco_path = Path(coco_path)
    if not coco_path.exists():
        raise FileNotFoundError(coco_path)

    d = json.load(open(coco_path))

    # 从 COCO 文件自身建立映射，不信任外部 CLASS_NAMES 索引
    cat_id_to_name = {c["id"]: c["name"] for c in d["categories"]}
    cat_name_to_id = {c["name"]: c["id"] for c in d["categories"]}

    image_id_to_name = {
        im["id"]: im["file_name"].rsplit(".", 1)[0]
        for im in d["images"]
    }

    errors = []
    data = {}

    for ann in d["annotations"]:
        img_name = image_id_to_name[ann["image_id"]]
        x, y, w, h = ann["bbox"]
        cat_id = ann["category_id"]

        if cat_id not in cat_id_to_name:
            errors.append(f"未知 category_id={cat_id}")
            continue

        if w <= 0 or h <= 0:
            errors.append(f"非法 bbox width/height ann_id={ann['id']}")
        if ann["iscrowd"] != 0:
            errors.append(f"iscrowd=1 ann_id={ann['id']}")

        # COCO → YOLO，保留 6 位小数
        cx = round((x + w / 2) / IMG_W, 6)
        cy = round((y + h / 2) / IMG_H, 6)
        nw = round(w / IMG_W, 6)
        nh = round(h / IMG_H, 6)

        if nw <= 0 or nh <= 0:
            errors.append(f"还原后 YOLO 宽高非法 ann_id={ann['id']}")

        data.setdefault(img_name, []).append({
            "name": cat_id_to_name[cat_id],
            "yolo": (cat_id - 1, cx, cy, nw, nh),
        })

    return data, errors, cat_id_to_name, cat_name_to_id

In [22]:
def verify(split):
    yolo = load_yolo_normalized(ROOT, split)
    coco, errors, cat_id_to_name, cat_name_to_id = load_coco_and_roundtrip(
        COCO_DIR / f"{split}.json"
    )

    yolo_names = set(yolo)
    coco_names = set(coco)

    print("="*65)
    print(f"split = {split}")
    print("="*65)
    print(f"YOLO 图片数：{len(yolo_names)}")
    print(f"COCO 图片数：{len(coco_names)}")
    print(f"只在 YOLO：{len(yolo_names - coco_names)}")
    print(f"只在 COCO：{len(coco_names - yolo_names)}")

    print("\n--- COCO 结构检查 ---")
    if errors:
        print("❌ 发现错误：")
        for e in errors[:20]:
            print("  ", e)
        return False
    print("✅ COCO bbox / area / iscrowd / category_id 合法")

    print("\n--- COCO 类别定义 ---")
    for cid, cname in sorted(cat_id_to_name.items()):
        print(f"  category_id={cid}  name={cname}")

    # 类别名称白名单检查
    expected_names = {"Person", "Car", "Bicycle", "OtherVehicle"}
    actual_names = set(cat_id_to_name.values())
    unknown = actual_names - expected_names
    if unknown:
        print(f"\n❌ 出现未预期类别：{unknown}")
        return False

    print("\n--- YOLO ↔ COCO→YOLO 往返比较 ---")

    max_diff = 0.0
    box_compared = 0
    class_error = 0
    box_num_error = 0

    for name in sorted(yolo):
        if name not in coco:
            continue

        yb = sorted(yolo[name])
        cb = sorted((x["yolo"][0], x["yolo"][1], x["yolo"][2], x["yolo"][3], x["yolo"][4]) for x in coco[name])

        if len(yb) != len(cb):
            box_num_error += 1
            print(f"❌ 框数不一致：{name} YOLO={len(yb)} COCO={len(cb)}")
            continue

        for y, c in zip(yb, cb):
            y_class, y_cx, y_cy, y_w, y_h = y
            c_class, c_cx, c_cy, c_w, c_h = c

            if y_class != c_class:
                class_error += 1
                continue

            diff = max(
                abs(y_cx - c_cx),
                abs(y_cy - c_cy),
                abs(y_w - c_w),
                abs(y_h - c_h),
            )
            max_diff = max(max_diff, diff)
            box_compared += 1

    print(f"比较框数：{box_compared}")
    print(f"框数不一致图片：{box_num_error}")
    print(f"类别不一致框数：{class_error}")
    print(f"最大 YOLO 归一化坐标差：{max_diff:.6f}")

    ok = (
        box_num_error == 0
        and class_error == 0
        and max_diff <= 1e-4
    )

    print("\n=== 最终结论 ===")
    if ok:
        print("✅ 转换通过：YOLO → COCO → YOLO 可无损往返，可用于训练")
    elif max_diff < 0.0001:
        print("⚠️ 差异很小，但仍需检查字段舍入规则")
    else:
        print("❌ 转换存在实质差异，禁止用于训练")

    return ok

In [23]:
verify("val")
verify("train")

split = val
YOLO 图片数：287
COCO 图片数：287
只在 YOLO：0
只在 COCO：0

--- COCO 结构检查 ---
✅ COCO bbox / area / iscrowd / category_id 合法

--- COCO 类别定义 ---
  category_id=1  name=Person
  category_id=2  name=Car
  category_id=3  name=Bicycle
  category_id=4  name=OtherVehicle

--- YOLO ↔ COCO→YOLO 往返比较 ---
比较框数：2453
框数不一致图片：0
类别不一致框数：0
最大 YOLO 归一化坐标差：0.000000

=== 最终结论 ===
✅ 转换通过：YOLO → COCO → YOLO 可无损往返，可用于训练
split = train
YOLO 图片数：2008
COCO 图片数：2008
只在 YOLO：0
只在 COCO：0

--- COCO 结构检查 ---
✅ COCO bbox / area / iscrowd / category_id 合法

--- COCO 类别定义 ---
  category_id=1  name=Person
  category_id=2  name=Car
  category_id=3  name=Bicycle
  category_id=4  name=OtherVehicle

--- YOLO ↔ COCO→YOLO 往返比较 ---
比较框数：17518
框数不一致图片：0
类别不一致框数：0
最大 YOLO 归一化坐标差：0.000000

=== 最终结论 ===
✅ 转换通过：YOLO → COCO → YOLO 可无损往返，可用于训练


True